<a href="https://colab.research.google.com/github/surjisudha/Game-Analytics-PowerBI/blob/main/Game_Analytics_%26_Sales_Intelligence_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

games = pd.read_csv("games.csv")
sales = pd.read_csv("vgsales.csv")

print("Games:", games.shape)
print("Sales:", sales.shape)

Games: (1512, 14)
Sales: (16598, 11)


Checking columns and data types

In [2]:
print("GAMES DATASET")
print(games.info())

print("\n" + "="*50)

print("SALES DATASET")
print(sales.info())

GAMES DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1512 entries, 0 to 1511
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1512 non-null   int64  
 1   Title              1512 non-null   object 
 2   Release Date       1512 non-null   object 
 3   Team               1511 non-null   object 
 4   Rating             1499 non-null   float64
 5   Times Listed       1512 non-null   object 
 6   Number of Reviews  1512 non-null   object 
 7   Genres             1512 non-null   object 
 8   Summary            1511 non-null   object 
 9   Reviews            1512 non-null   object 
 10  Plays              1512 non-null   object 
 11  Playing            1512 non-null   object 
 12  Backlogs           1512 non-null   object 
 13  Wishlist           1512 non-null   object 
dtypes: float64(1), int64(1), object(12)
memory usage: 165.5+ KB
None

SALES DATASET
<class 'pandas.core.frame.

Checking missing values

In [3]:
print("Missing values in games.csv:")
print(games.isnull().sum())

print("\nMissing values in vgsales.csv:")
print(sales.isnull().sum())

Missing values in games.csv:
Unnamed: 0            0
Title                 0
Release Date          0
Team                  1
Rating               13
Times Listed          0
Number of Reviews     0
Genres                0
Summary               1
Reviews               0
Plays                 0
Playing               0
Backlogs              0
Wishlist              0
dtype: int64

Missing values in vgsales.csv:
Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64


Checking duplicates

In [4]:
print("Duplicate rows in games.csv:", games.duplicated().sum())
print("Duplicate rows in vgsales.csv:", sales.duplicated().sum())

Duplicate rows in games.csv: 0
Duplicate rows in vgsales.csv: 0


Remove unnecessary column

In [5]:
games = games.drop("Unnamed: 0", axis=1)

print(games.columns)

Index(['Title', 'Release Date', 'Team', 'Rating', 'Times Listed',
       'Number of Reviews', 'Genres', 'Summary', 'Reviews', 'Plays', 'Playing',
       'Backlogs', 'Wishlist'],
      dtype='object')


Handle missing values

In [6]:
games["Team"] = games["Team"].fillna("Unknown")
games["Rating"] = games["Rating"].fillna(games["Rating"].median())
games["Summary"] = games["Summary"].fillna("No Summary")

sales["Publisher"] = sales["Publisher"].fillna("Unknown")
sales["Year"] = sales["Year"].fillna(0)

print("Missing values handled successfully!")

Missing values handled successfully!


Check that missing values are handled

In [7]:
print("Games missing values:")
print(games.isnull().sum())

print("\nSales missing values:")
print(sales.isnull().sum())

Games missing values:
Title                0
Release Date         0
Team                 0
Rating               0
Times Listed         0
Number of Reviews    0
Genres               0
Summary              0
Reviews              0
Plays                0
Playing              0
Backlogs             0
Wishlist             0
dtype: int64

Sales missing values:
Rank            0
Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64


Convert engagement columns (k to num)

In [8]:
def convert_to_number(value):
    value = str(value).strip()

    if value.endswith("K"):
        return float(value[:-1]) * 1000

    elif value.endswith("M"):
        return float(value[:-1]) * 1000000

    else:
        return float(value)


columns = [
    "Times Listed",
    "Number of Reviews",
    "Plays",
    "Playing",
    "Backlogs",
    "Wishlist"
]

for column in columns:
    games[column] = games[column].apply(convert_to_number)

print(games[columns].head())

   Times Listed  Number of Reviews    Plays  Playing  Backlogs  Wishlist
0        3900.0             3900.0  17000.0   3800.0    4600.0    4800.0
1        2900.0             2900.0  21000.0   3200.0    6300.0    3600.0
2        4300.0             4300.0  30000.0   2500.0    5000.0    2600.0
3        3500.0             3500.0  28000.0    679.0    4900.0    1800.0
4        3000.0             3000.0  21000.0   2400.0    8300.0    2300.0


Verifying the data types

In [9]:
print(games[columns].dtypes)

Times Listed         float64
Number of Reviews    float64
Plays                float64
Playing              float64
Backlogs             float64
Wishlist             float64
dtype: object


Now we'll convert Release Date from text to an actual date and create a new Release Year column.

In [10]:
games["Release Date"] = pd.to_datetime(
    games["Release Date"],
    errors="coerce"
)

games["Release Year"] = games["Release Date"].dt.year

print(games[["Title", "Release Date", "Release Year"]].head())

                                     Title Release Date  Release Year
0                               Elden Ring   2022-02-25        2022.0
1                                    Hades   2019-12-10        2019.0
2  The Legend of Zelda: Breath of the Wild   2017-03-03        2017.0
3                                Undertale   2015-09-15        2015.0
4                            Hollow Knight   2017-02-24        2017.0


In [11]:
print(games["Release Date"].dtype)
print("\nMissing Release Dates:", games["Release Date"].isnull().sum())

datetime64[ns]

Missing Release Dates: 3


Handling the 3 missing release dates

In [12]:
games["Release Year"] = games["Release Year"].fillna(0).astype(int)

print("Missing Release Dates:", games["Release Date"].isnull().sum())
print("Missing Release Years:", games["Release Year"].isnull().sum())

print(games[["Title", "Release Date", "Release Year"]].head())

Missing Release Dates: 3
Missing Release Years: 0
                                     Title Release Date  Release Year
0                               Elden Ring   2022-02-25          2022
1                                    Hades   2019-12-10          2019
2  The Legend of Zelda: Breath of the Wild   2017-03-03          2017
3                                Undertale   2015-09-15          2015
4                            Hollow Knight   2017-02-24          2017


Clean the Sales Dataset

In [13]:
print(sales[["Year", "Publisher", "Genre", "Platform"]].head(10))

     Year Publisher         Genre Platform
0  2006.0  Nintendo        Sports      Wii
1  1985.0  Nintendo      Platform      NES
2  2008.0  Nintendo        Racing      Wii
3  2009.0  Nintendo        Sports      Wii
4  1996.0  Nintendo  Role-Playing       GB
5  1989.0  Nintendo        Puzzle       GB
6  2006.0  Nintendo      Platform       DS
7  2006.0  Nintendo          Misc      Wii
8  2009.0  Nintendo      Platform      Wii
9  1984.0  Nintendo       Shooter      NES


In [14]:
print("Unique Platforms:", sales["Platform"].nunique())
print("Unique Genres:", sales["Genre"].nunique())
print("Unique Publishers:", sales["Publisher"].nunique())

Unique Platforms: 31
Unique Genres: 12
Unique Publishers: 578


In [15]:
print(sales.describe())

               Rank          Year      NA_Sales      EU_Sales      JP_Sales  \
count  16598.000000  16598.000000  16598.000000  16598.000000  16598.000000   
mean    8300.605254   1973.647307      0.264667      0.146652      0.077782   
std     4791.853933    254.346809      0.816683      0.505351      0.309291   
min        1.000000      0.000000      0.000000      0.000000      0.000000   
25%     4151.250000   2003.000000      0.000000      0.000000      0.000000   
50%     8300.500000   2007.000000      0.080000      0.020000      0.000000   
75%    12449.750000   2010.000000      0.240000      0.110000      0.040000   
max    16600.000000   2020.000000     41.490000     29.020000     10.220000   

        Other_Sales  Global_Sales  
count  16598.000000  16598.000000  
mean       0.048063      0.537441  
std        0.188588      1.555028  
min        0.000000      0.010000  
25%        0.000000      0.060000  
50%        0.010000      0.170000  
75%        0.040000      0.470000  


Fix the Year column

In [16]:
sales["Year"] = sales["Year"].replace(0, pd.NA)

print("Missing Years:", sales["Year"].isna().sum())
print(sales["Year"].describe())

Missing Years: 271
count     16327.0
unique       39.0
top        2009.0
freq       1431.0
Name: Year, dtype: float64


Clean text columns

Now let's standardize the three important categorical columns:

Platform, Genre, Publisher

In [17]:
# Remove extra spaces
sales["Name"] = sales["Name"].str.strip()
sales["Platform"] = sales["Platform"].str.strip()
sales["Genre"] = sales["Genre"].str.strip()
sales["Publisher"] = sales["Publisher"].str.strip()

# Replace empty strings with Unknown
sales["Name"] = sales["Name"].replace("", "Unknown")
sales["Platform"] = sales["Platform"].replace("", "Unknown")
sales["Genre"] = sales["Genre"].replace("", "Unknown")
sales["Publisher"] = sales["Publisher"].replace("", "Unknown")

# Check unique values
print("Platforms:", sales["Platform"].nunique())
print("Genres:", sales["Genre"].nunique())
print("Publishers:", sales["Publisher"].nunique())

Platforms: 31
Genres: 12
Publishers: 578


In [18]:
print("\nPlatforms:")
print(sorted(sales["Platform"].unique()))

print("\nGenres:")
print(sorted(sales["Genre"].unique())[:30])


Platforms:
['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG', 'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360', 'XB', 'XOne']

Genres:
['Action', 'Adventure', 'Fighting', 'Misc', 'Platform', 'Puzzle', 'Racing', 'Role-Playing', 'Shooter', 'Simulation', 'Sports', 'Strategy']


 Final validation

In [19]:
print("GAMES DATASET")
print(games.info())

print("\n" + "="*60)

print("SALES DATASET")
print(sales.info())

GAMES DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1512 entries, 0 to 1511
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Title              1512 non-null   object        
 1   Release Date       1509 non-null   datetime64[ns]
 2   Team               1512 non-null   object        
 3   Rating             1512 non-null   float64       
 4   Times Listed       1512 non-null   float64       
 5   Number of Reviews  1512 non-null   float64       
 6   Genres             1512 non-null   object        
 7   Summary            1512 non-null   object        
 8   Reviews            1512 non-null   object        
 9   Plays              1512 non-null   float64       
 10  Playing            1512 non-null   float64       
 11  Backlogs           1512 non-null   float64       
 12  Wishlist           1512 non-null   float64       
 13  Release Year       1512 non-null   int64         

In [20]:
print("\nGames missing values:")
print(games.isnull().sum())

print("\nSales missing values:")
print(sales.isnull().sum())


Games missing values:
Title                0
Release Date         3
Team                 0
Rating               0
Times Listed         0
Number of Reviews    0
Genres               0
Summary              0
Reviews              0
Plays                0
Playing              0
Backlogs             0
Wishlist             0
Release Year         0
dtype: int64

Sales missing values:
Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher         0
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64


In [21]:
games.to_csv("clean_games.csv", index=False)
sales.to_csv("clean_vgsales.csv", index=False)

print("\n✅ Cleaned datasets saved successfully!")
print("1. clean_games.csv")
print("2. clean_vgsales.csv")


✅ Cleaned datasets saved successfully!
1. clean_games.csv
2. clean_vgsales.csv


Now we'll use Python to explore the cleaned data and generate actual insights.

EDA— Basic Statistics & Distribution.
Basic statistics

In [22]:
print("GAMES DATASET")
print(games.describe())

print("\n" + "="*60)

print("SALES DATASET")
print(sales.describe())

GAMES DATASET
                        Release Date       Rating  Times Listed  \
count                           1509  1512.000000   1512.000000   
mean   2012-09-16 19:03:13.240556800     3.720040    769.459656   
min              1980-05-22 00:00:00     0.700000      0.000000   
25%              2007-09-16 00:00:00     3.400000    284.000000   
50%              2014-09-01 00:00:00     3.800000    551.000000   
75%              2019-09-13 00:00:00     4.100000   1000.000000   
max              2025-03-31 00:00:00     4.800000   4300.000000   
std                              NaN     0.530364    687.840871   

       Number of Reviews         Plays      Playing     Backlogs     Wishlist  \
count        1512.000000   1512.000000  1512.000000  1512.000000  1512.000000   
mean          769.459656   6253.578704   267.379630  1452.577381   780.540344   
min             0.000000      0.000000     0.000000     1.000000     2.000000   
25%           284.000000   1800.000000    43.000000   461.

In [23]:
print("Games:", len(games))
print("Unique game titles:", games["Title"].nunique())
print("Unique developers:", games["Team"].nunique())
print("Unique genres:", games["Genres"].nunique())

print("\nSales:", len(sales))
print("Unique games:", sales["Name"].nunique())
print("Unique platforms:", sales["Platform"].nunique())
print("Unique genres:", sales["Genre"].nunique())
print("Unique publishers:", sales["Publisher"].nunique())

Games: 1512
Unique game titles: 1099
Unique developers: 765
Unique genres: 255

Sales: 16598
Unique games: 11493
Unique platforms: 31
Unique genres: 12
Unique publishers: 578


In [24]:
print("Top 10 Most Wishlisted Games")
print(
    games[["Title", "Wishlist"]]
    .sort_values("Wishlist", ascending=False)
    .head(10)
)

Top 10 Most Wishlisted Games
                                         Title  Wishlist
972  The Legend of Zelda: Tears of the Kingdom    5400.0
326                                 Elden Ring    4800.0
0                                   Elden Ring    4800.0
776                                 Elden Ring    4800.0
6                                        Omori    3800.0
332                                      Omori    3800.0
782                                      Omori    3800.0
31         NieR Replicant ver.1.22474487139...    3700.0
357        NieR Replicant ver.1.22474487139...    3700.0
807        NieR Replicant ver.1.22474487139...    3700.0


let's create an EDA-safe copy of the games data.

In [25]:
games_eda = games.copy()

# Treat unknown release year as missing
games_eda["Release Year"] = games_eda["Release Year"].replace(0, pd.NA)

print("EDA dataset shape:", games_eda.shape)
print("Unknown release years:", games_eda["Release Year"].isna().sum())

EDA dataset shape: (1512, 14)
Unknown release years: 3


Remove duplicate game titles for title-level analysis
We won't delete these records from the original dataset.
Instead, we'll create a separate title-level dataset:

In [26]:
games_unique = games_eda.drop_duplicates(subset="Title").copy()

print("Original rows:", len(games_eda))
print("Unique game titles:", len(games_unique))

Original rows: 1512
Unique game titles: 1099


Correct Top 10 Wishlisted Games

In [27]:
top_wishlist = (
    games_unique[["Title", "Wishlist"]]
    .sort_values("Wishlist", ascending=False)
    .head(10)
)

print(top_wishlist.to_string(index=False))

                                    Title  Wishlist
The Legend of Zelda: Tears of the Kingdom    5400.0
                               Elden Ring    4800.0
                                    Omori    3800.0
      NieR Replicant ver.1.22474487139...    3700.0
                           NieR: Automata    3600.0
                                    Hades    3600.0
                                    Stray    3400.0
                Sekiro: Shadows Die Twice    3400.0
                      God of War Ragnarök    3300.0
                            Metroid Dread    3300.0


EDA Questions:
Q1 : What are the top-rated games by user reviews?

In [28]:
top_rated = (
    games_unique[["Title", "Rating"]]
    .sort_values("Rating", ascending=False)
    .head(10)
)

print(top_rated.to_string(index=False))

                                   Title  Rating
       Elden Ring: Shadow of the Erdtree     4.8
    Bloodborne: Game of the Year Edition     4.6
            Disco Elysium: The Final Cut     4.6
                             Tokyo Necro     4.6
                             Outer Wilds     4.6
       Final Fantasy XIV: Shadowbringers     4.6
       The Great Ace Attorney 2: Resolve     4.6
Sekiro: Shadows Die Twice - GOTY Edition     4.6
                         Half-Life: Alyx     4.6
                          Dwarf Fortress     4.6


Q2. Which developers (Teams) have the highest average ratings?

In [29]:
top_developers = (
    games.groupby("Team")["Rating"]
    .agg(["mean", "count"])
    .reset_index()
)

top_developers = top_developers[
    top_developers["count"] >= 3
].sort_values("mean", ascending=False).head(10)

print(top_developers.to_string(index=False))

                                                    Team     mean  count
                                               ['ZA/UM'] 4.600000      6
             ['Mobius Digital', 'Annapurna Interactive'] 4.575000      4
         ['FromSoftware', 'Sony Computer Entertainment'] 4.540000      5
       ['Konami Computer Entertainment Japan', 'Konami'] 4.500000      5
                               ['Team Silent', 'Konami'] 4.500000      4
                        ['Square Enix', 'PlatinumGames'] 4.466667      3
                                         ['Team Cherry'] 4.440000      5
                    ['Buena Vista Games', 'Square Enix'] 4.433333      3
                          ['FromSoftware', 'Activision'] 4.400000      3
['Sony Computer Entertainment, Inc. (SCEI)', 'Team Ico'] 4.400000      3


Q3. What are the most common genres in the dataset?

In [30]:
# Split multiple genres and count each genre

genre_counts = (
    games["Genres"]
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

print("Most Common Genres:")
print(genre_counts)

Most Common Genres:
Genres
['Adventure'        970
'RPG']              251
'Shooter']          235
'Indie'             193
'RPG'               177
                   ... 
['Visual Novel']      2
['Strategy'           1
['MOBA']              1
['Indie']             1
'Pinball']            1
Name: count, Length: 69, dtype: int64


In [34]:
import ast

def extract_genres(value):
    try:
        return ast.literal_eval(value)
    except:
        return [value]

genre_counts = (
    games["Genres"]
    .apply(extract_genres)
    .explode()
    .str.strip()
    .value_counts()
)

print("Most Common Genres:")
print(genre_counts)

Most Common Genres:
Genres
Adventure              1014
RPG                     523
Shooter                 353
Platform                329
Indie                   284
Puzzle                  176
Brawler                 159
Strategy                142
Simulator               125
Turn Based Strategy      99
Arcade                   73
Fighting                 72
Visual Novel             71
Tactical                 50
Point-and-Click          46
Racing                   42
Music                    25
Sport                    25
Card & Board Game        16
Real Time Strategy       10
Quiz/Trivia               4
MOBA                      3
Pinball                   1
Name: count, dtype: int64


In [31]:
top_genres = genre_counts.head(10)

print("\nTop 10 Most Common Genres:")
print(top_genres)


Top 10 Most Common Genres:
Genres
['Adventure'              970
'RPG']                    251
'Shooter']                235
'Indie'                   193
'RPG'                     177
'Platform']               168
'Platform'                114
'Turn Based Strategy']     99
'Strategy']                89
'Puzzle']                  85
Name: count, dtype: int64


Q4. Which games have the highest backlog compared to wishlist?

In [32]:
games_unique["Backlog_Wishlist_Ratio"] = (
    games_unique["Backlogs"] / games_unique["Wishlist"]
)

In [33]:
top_backlog = (
    games_unique[
        ["Title", "Backlogs", "Wishlist", "Backlog_Wishlist_Ratio"]
    ]
    .sort_values("Backlog_Wishlist_Ratio", ascending=False)
    .head(10)
)

print(top_backlog.to_string(index=False))

                      Title  Backlogs  Wishlist  Backlog_Wishlist_Ratio
                   Paladins     188.0      16.0               11.750000
                    Figment     392.0      35.0               11.200000
                   Fortnite     470.0      47.0               10.000000
                 Pokémon Go     197.0      22.0                8.954545
  Half-Life: Opposing Force     838.0      95.0                8.821053
                  Destiny 2     845.0     103.0                8.203883
                 Brawlhalla     294.0      36.0                8.166667
      BioShock 2 Remastered    2000.0     246.0                8.130081
Borderlands: The Pre-Sequel    1900.0     238.0                7.983193
        PUBG: Battlegrounds     253.0      32.0                7.906250


Q5 — What is the game release trend across years?

In [35]:
release_trend = (
    games_eda[games_eda["Release Year"].notna()]
    .groupby("Release Year")
    .size()
    .reset_index(name="Game Count")
    .sort_values("Release Year")
)

print(release_trend.to_string(index=False))

 Release Year  Game Count
         1980           1
         1982           1
         1985           1
         1986           4
         1987           3
         1988           5
         1989           5
         1990           6
         1991          11
         1992           6
         1993          12
         1994          11
         1995          12
         1996          15
         1997          18
         1998          16
         1999          25
         2000          15
         2001          30
         2002          22
         2003          26
         2004          37
         2005          41
         2006          32
         2007          50
         2008          44
         2009          40
         2010          58
         2011          50
         2012          60
         2013          68
         2014          59
         2015          73
         2016          75
         2017          70
         2018          77
         2019          87
         202

Q6 — Distribution of user ratings

In [36]:
rating_distribution = (
    games_unique["Rating"]
    .value_counts()
    .sort_index()
)

print(rating_distribution)

Rating
0.7     1
1.2     1
1.4     1
1.5     1
1.6     2
1.8     1
1.9     2
2.0     1
2.1     3
2.2     2
2.3     5
2.4     8
2.5    13
2.6     8
2.7    15
2.8    22
2.9    24
3.0    28
3.1    39
3.2    45
3.3    46
3.4    58
3.5    76
3.6    83
3.7    96
3.8    86
3.9    75
4.0    86
4.1    78
4.2    70
4.3    54
4.4    32
4.5    20
4.6    16
4.8     1
Name: count, dtype: int64


Q8. What is the average number of plays per genre?

In [37]:
games_genres = games.copy()

games_genres["Genres"] = games_genres["Genres"].apply(extract_genres)

games_genres = games_genres.explode("Genres")

games_genres["Genres"] = games_genres["Genres"].str.strip()

avg_plays_genre = (
    games_genres
    .groupby("Genres")["Plays"]
    .mean()
    .sort_values(ascending=False)
)

print("Average Plays per Genre:")
print(avg_plays_genre)

Average Plays per Genre:
Genres
MOBA                   10166.666667
Shooter                 7522.872521
Racing                  7197.000000
Platform                7162.522796
Turn Based Strategy     6834.181818
Adventure               6546.182446
Brawler                 6436.415094
RPG                     6296.061185
Sport                   5977.320000
Simulator               5738.080000
Indie                   5461.010563
Puzzle                  5296.772727
Arcade                  5161.000000
Fighting                5112.486111
Strategy                4920.838028
Tactical                4476.980000
Point-and-Click         4028.630435
Real Time Strategy      3860.000000
Card & Board Game       3609.187500
Visual Novel            3572.295775
Music                   2715.240000
Quiz/Trivia             2525.000000
Pinball                 1600.000000
Name: Plays, dtype: float64


Q9. Which developer studios are the most productive and impactful?

In [38]:
developer_analysis = (
    games.groupby("Team")
    .agg(
        Game_Count=("Title", "nunique"),
        Average_Rating=("Rating", "mean"),
        Average_Plays=("Plays", "mean"),
        Total_Wishlist=("Wishlist", "sum")
    )
    .reset_index()
)

# Focus on developers with at least 3 games
developer_analysis = developer_analysis[
    developer_analysis["Game_Count"] >= 3
]

# Sort by productivity and engagement
developer_analysis = developer_analysis.sort_values(
    ["Game_Count", "Average_Plays"],
    ascending=False
)

print(developer_analysis.head(15).to_string(index=False))

                                           Team  Game_Count  Average_Rating  Average_Plays  Total_Wishlist
                                     ['Capcom']          26        3.722857    4645.057143         22777.0
                                ['Square Enix']          22        3.941935    3420.709677         32098.0
                     ['Nintendo', 'Game Freak']          12        3.805263    9415.789474         11156.0
                   ['Nintendo', 'Nintendo EAD']          11        3.842105   13552.631579         11618.0
                                   ['Nintendo']          11        3.810526   10068.842105         12478.0
  ['Intelligent Systems Co., Ltd.', 'Nintendo']          11        3.550000    3592.000000          6259.0
                   ['Nintendo EAD', 'Nintendo']           9        3.887500   11396.250000         11050.0
                                     ['Konami']           9        3.644444    1783.000000          2692.0
  ['Ubisoft Montreal', 'Ubisoft Enter

Sales Analysis
Q10. Which region generates the most game sales?

In [39]:
regional_sales = {
    "North America": sales["NA_Sales"].sum(),
    "Europe": sales["EU_Sales"].sum(),
    "Japan": sales["JP_Sales"].sum(),
    "Other": sales["Other_Sales"].sum()
}

regional_sales = pd.Series(regional_sales).sort_values(ascending=False)

print("Regional Sales (Millions):")
print(regional_sales)

Regional Sales (Millions):
North America    4392.95
Europe           2434.13
Japan            1291.02
Other             797.75
dtype: float64


Q11 — What are the best-selling platforms?

In [40]:
platform_sales = (
    sales.groupby("Platform")["Global_Sales"]
    .sum()
    .sort_values(ascending=False)
)

print("Best-Selling Platforms:")
print(platform_sales)

Best-Selling Platforms:
Platform
PS2     1255.64
X360     979.96
PS3      957.84
Wii      926.71
DS       822.49
PS       730.66
GBA      318.50
PSP      296.28
PS4      278.10
PC       258.82
XB       258.26
GB       255.45
NES      251.07
3DS      247.46
N64      218.88
SNES     200.05
GC       199.36
XOne     141.06
2600      97.08
WiiU      81.86
PSV       61.93
SAT       33.59
GEN       28.36
DC        15.97
SCD        1.87
NG         1.44
WS         1.42
TG16       0.16
3DO        0.10
GG         0.04
PCFX       0.03
Name: Global_Sales, dtype: float64


Q12 — Trend of game releases and sales over years

In [41]:
yearly_sales = (
    sales.dropna(subset=["Year"])
    .groupby("Year")
    .agg(
        Game_Releases=("Name", "count"),
        Global_Sales=("Global_Sales", "sum")
    )
    .reset_index()
    .sort_values("Year")
)

print(yearly_sales.to_string(index=False))

  Year  Game_Releases  Global_Sales
1980.0              9         11.38
1981.0             46         35.77
1982.0             36         28.86
1983.0             17         16.79
1984.0             14         50.36
1985.0             14         53.94
1986.0             21         37.07
1987.0             16         21.74
1988.0             15         47.22
1989.0             17         73.45
1990.0             16         49.39
1991.0             41         32.23
1992.0             43         76.16
1993.0             60         45.98
1994.0            121         79.17
1995.0            219         88.11
1996.0            263        199.15
1997.0            289        200.98
1998.0            379        256.47
1999.0            338        251.27
2000.0            349        201.56
2001.0            482        331.47
2002.0            829        395.52
2003.0            775        357.85
2004.0            763        419.31
2005.0            941        459.94
2006.0           1008       

Q13 — Who are the top publishers by sales?

In [42]:
publisher_sales = (
    sales.groupby("Publisher")["Global_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("Top 10 Publishers by Global Sales:")
print(publisher_sales)

Top 10 Publishers by Global Sales:
Publisher
Nintendo                        1786.56
Electronic Arts                 1110.32
Activision                       727.46
Sony Computer Entertainment      607.50
Ubisoft                          474.72
Take-Two Interactive             399.54
THQ                              340.77
Konami Digital Entertainment     283.64
Sega                             272.99
Namco Bandai Games               254.09
Name: Global_Sales, dtype: float64


Q14 — Which games are the top 10 best-sellers globally?

In [43]:
top_games_sales = (
    sales.groupby("Name")["Global_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("Top 10 Best-Selling Games Globally:")
print(top_games_sales)

Top 10 Best-Selling Games Globally:
Name
Wii Sports                        82.74
Grand Theft Auto V                55.92
Super Mario Bros.                 45.31
Tetris                            35.84
Mario Kart Wii                    35.82
Wii Sports Resort                 33.00
Pokemon Red/Pokemon Blue          31.37
Call of Duty: Black Ops           31.03
Call of Duty: Modern Warfare 3    30.83
New Super Mario Bros.             30.01
Name: Global_Sales, dtype: float64


Q15 — How do regional sales compare for specific platforms?

In [44]:
platform_region_sales = (
    sales.groupby("Platform")[
        ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
    ]
    .sum()
    .sort_values("NA_Sales", ascending=False)
)

print("Regional Sales by Platform:")
print(platform_region_sales)

Regional Sales by Platform:
          NA_Sales  EU_Sales  JP_Sales  Other_Sales
Platform                                           
X360        601.05    280.58     12.43        85.54
PS2         583.84    339.29    139.20       193.44
Wii         507.71    268.38     69.35        80.61
PS3         392.26    343.71     79.99       141.93
DS          390.71    194.65    175.57        60.53
PS          336.51    213.60    139.82        40.91
GBA         187.54     75.25     47.33         7.73
XB          186.69     60.95      1.38         8.72
N64         139.02     41.06     34.22         4.38
GC          133.46     38.71     21.58         5.18
NES         125.94     21.15     98.65         5.31
GB          114.32     47.82     85.12         8.20
PSP         108.99     68.25     76.79        42.19
PS4          96.80    123.70     14.30        43.36
PC           93.28    139.68      0.17        24.86
2600         90.60      5.47      0.00         0.91
XOne         83.19     45.65      0.

Q16 — How has the market evolved by platform over time?

In [45]:
platform_year_sales = (
    sales.dropna(subset=["Year"])
    .groupby(["Year", "Platform"])["Global_Sales"]
    .sum()
    .reset_index()
)

print(platform_year_sales.head(30).to_string(index=False))

  Year Platform  Global_Sales
1980.0     2600         11.38
1981.0     2600         35.77
1982.0     2600         28.86
1983.0     2600          5.83
1983.0      NES         10.96
1984.0     2600          0.27
1984.0      NES         50.09
1985.0     2600          0.45
1985.0       DS          0.02
1985.0      NES         53.44
1985.0       PC          0.03
1986.0     2600          0.66
1986.0      NES         36.41
1987.0     2600          1.98
1987.0      NES         19.76
1988.0     2600          0.75
1988.0       GB          1.43
1988.0      NES         45.01
1988.0       PC          0.03
1989.0     2600          0.62
1989.0       GB         64.98
1989.0      NES          7.85
1990.0       GB          4.89
1990.0      GEN          2.60
1990.0      NES         15.74
1990.0     SNES         26.16
1991.0       GB          5.57
1991.0      GEN          4.34
1991.0      NES          6.11
1991.0     SNES         16.21


Q17 — What are the regional genre preferences?

In [46]:
genre_region_sales = (
    sales.groupby("Genre")[
        ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
    ]
    .sum()
    .sort_values("NA_Sales", ascending=False)
)

print("Regional Sales by Genre:")
print(genre_region_sales)

Regional Sales by Genre:
              NA_Sales  EU_Sales  JP_Sales  Other_Sales
Genre                                                  
Action          877.83    525.00    159.95       187.38
Sports          683.35    376.85    135.37       134.97
Shooter         582.60    313.27     38.28       102.69
Platform        447.05    201.63    130.77        51.59
Misc            410.24    215.98    107.76        75.32
Racing          359.42    238.39     56.69        77.27
Role-Playing    327.28    188.06    352.31        59.61
Fighting        223.59    101.32     87.35        36.68
Simulation      183.31    113.38     63.70        31.52
Puzzle          123.78     50.78     57.31        12.55
Adventure       105.80     64.13     52.07        16.81
Strategy         68.70     45.34     49.46        11.36


Q18 — What's the yearly sales change per region?

In [47]:
yearly_region_sales = (
    sales.dropna(subset=["Year"])
    .groupby("Year")[
        ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
    ]
    .sum()
    .reset_index()
    .sort_values("Year")
)

print(yearly_region_sales.to_string(index=False))

  Year  NA_Sales  EU_Sales  JP_Sales  Other_Sales
1980.0     10.59      0.67      0.00         0.12
1981.0     33.40      1.96      0.00         0.32
1982.0     26.92      1.65      0.00         0.31
1983.0      7.76      0.80      8.10         0.14
1984.0     33.28      2.10     14.27         0.70
1985.0     33.73      4.74     14.56         0.92
1986.0     12.50      2.84     19.81         1.93
1987.0      8.46      1.41     11.63         0.20
1988.0     23.87      6.59     15.76         0.99
1989.0     45.15      8.44     18.36         1.50
1990.0     25.46      7.63     14.88         1.40
1991.0     12.76      3.95     14.78         0.74
1992.0     33.87     11.71     28.91         1.65
1993.0     15.12      4.65     25.33         0.89
1994.0     28.15     14.88     33.99         2.20
1995.0     24.82     14.90     45.75         2.64
1996.0     86.76     47.26     57.44         7.69
1997.0     94.75     48.32     48.87         9.13
1998.0    128.36     66.90     50.04        11.03


Q19 — What is the average sales per publisher?

In [48]:
avg_sales_publisher = (
    sales.groupby("Publisher")["Global_Sales"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

print("Top 10 Publishers by Average Sales per Game:")
print(avg_sales_publisher)

Top 10 Publishers by Average Sales per Game:
Publisher
Palcom                                4.170000
Red Orb                               2.620000
Nintendo                              2.541337
Arena Entertainment                   2.360000
UEP Systems                           2.250000
RedOctane                             2.170000
Valve                                 1.740000
Hello Games                           1.600000
Sony Computer Entertainment Europe    1.592667
Westwood Studios                      1.550000
Name: Global_Sales, dtype: float64


Q20 — Top 5 best-selling games per platform

In [49]:
top5_per_platform = (
    sales.sort_values(
        ["Platform", "Global_Sales"],
        ascending=[True, False]
    )
    .groupby("Platform")
    .head(5)
)

print(
    top5_per_platform[
        ["Platform", "Name", "Global_Sales"]
    ].to_string(index=False)
)

Platform                                             Name  Global_Sales
    2600                                          Pac-Man          7.81
    2600                                         Pitfall!          4.50
    2600                                        Asteroids          4.31
    2600                                  Missile Command          2.76
    2600                                   Space Invaders          2.53
     3DO                                      Policenauts          0.06
     3DO                                      Bust-A-Move          0.02
     3DO             Sotsugyou II: Neo Generation Special          0.02
     3DS                              Pokemon X/Pokemon Y         14.35
     3DS                                     Mario Kart 7         12.21
     3DS        Pokemon Omega Ruby/Pokemon Alpha Sapphire         11.33
     3DS                              Super Mario 3D Land         10.79
     3DS                          New Super Mario Bros. 2       

Merged Dataset.

In [50]:
merged = pd.merge(
    games,
    sales,
    left_on="Title",
    right_on="Name",
    how="inner"
)

print("Merged Dataset Shape:", merged.shape)
print(merged[["Title", "Genre", "Global_Sales", "Rating", "Plays", "Wishlist"]].head())

Merged Dataset Shape: (1344, 25)
       Title Genre  Global_Sales  Rating    Plays  Wishlist
0  Minecraft  Misc          9.20     4.3  33000.0     230.0
1  Minecraft  Misc          5.42     4.3  33000.0     230.0
2  Minecraft  Misc          4.02     4.3  33000.0     230.0
3  Minecraft  Misc          2.41     4.3  33000.0     230.0
4  Minecraft  Misc          2.25     4.3  33000.0     230.0


Q21 — Which game genres generate the most global sales?

In [51]:
genre_global_sales = (
    merged.groupby("Genre")["Global_Sales"]
    .sum()
    .sort_values(ascending=False)
)

print("Global Sales by Genre:")
print(genre_global_sales)

Global Sales by Genre:
Genre
Action          960.51
Platform        643.09
Shooter         532.48
Role-Playing    396.83
Sports          279.60
Racing          198.01
Puzzle          135.33
Misc            133.87
Fighting         87.06
Adventure        65.04
Simulation       35.06
Strategy         13.22
Name: Global_Sales, dtype: float64


Q22 — How does user rating affect global sales?

In [52]:
rating_sales = (
    merged.groupby("Rating")["Global_Sales"]
    .agg(["mean", "sum", "count"])
    .reset_index()
    .sort_values("Rating")
)

print("Rating vs Global Sales:")
print(rating_sales.to_string(index=False))

Rating vs Global Sales:
 Rating     mean    sum  count
    0.7 0.410000   0.41      1
    1.4 0.580000   0.58      1
    1.6 0.512500   2.05      4
    1.8 0.270000   0.27      1
    1.9 2.770000   5.54      2
    2.1 0.425000   0.85      2
    2.3 0.420000   0.42      1
    2.4 3.207778  28.87      9
    2.5 2.008500  40.17     20
    2.6 1.642500   6.57      4
    2.7 0.606250   4.85      8
    2.8 1.458889  39.39     27
    2.9 2.446957  56.28     23
    3.0 1.554318  68.39     44
    3.1 1.888793 109.55     58
    3.2 1.753462 136.77     78
    3.3 1.298889  70.14     54
    3.4 1.214507  86.23     71
    3.5 3.584524 301.10     84
    3.6 1.654141 163.76     99
    3.7 4.838641 498.38    103
    3.8 4.558090 405.67     89
    3.9 2.851776 305.14    107
    4.0 1.817923 236.33    130
    4.1 3.348161 291.29     87
    4.2 3.152667 236.45     75
    4.3 2.697672 312.93    116
    4.4 1.308667  39.26     30
    4.5 2.127333  31.91     15
    4.6 0.550000   0.55      1


Q23 — Which platforms have the most games with high ratings (>4)?

In [53]:
high_rating_platforms = (
    merged[merged["Rating"] > 4]
    .groupby("Platform")["Title"]
    .count()
    .sort_values(ascending=False)
)

print("Platforms with Games Rated Above 4:")
print(high_rating_platforms)

Platforms with Games Rated Above 4:
Platform
PS3     41
PS2     37
PC      35
X360    30
PS4     22
PS      21
DS      20
SNES    19
GC      16
Wii     12
N64     12
3DS     12
XOne    11
WiiU    10
XB       7
PSV      6
GBA      6
GB       3
NES      2
PSP      1
SAT      1
Name: Title, dtype: int64


Q24 — What’s the trend of releases and sales over time?

In [54]:
merged_yearly = (
    merged.dropna(subset=["Year"])
    .groupby("Year")
    .agg(
        Game_Releases=("Title", "nunique"),
        Global_Sales=("Global_Sales", "sum"),
        Avg_Rating=("Rating", "mean"),
        Total_Wishlist=("Wishlist", "sum")
    )
    .reset_index()
    .sort_values("Year")
)

print("Merged Dataset Yearly Trend:")
print(merged_yearly.to_string(index=False))

Merged Dataset Yearly Trend:
  Year  Game_Releases  Global_Sales  Avg_Rating  Total_Wishlist
1981.0              1          1.65    3.600000            38.0
1982.0              1          7.81    3.400000            31.0
1984.0              1          1.22    3.400000            31.0
1985.0              1         40.24    3.500000           237.0
1986.0              2         14.25    3.266667          1291.0
1987.0              2          5.19    2.700000           442.0
1988.0              4         79.06    3.833333          1754.0
1989.0              2         91.63    3.950000           370.0
1990.0              3         23.18    3.566667           890.0
1991.0              4         24.63    3.471429          3015.0
1992.0              8         40.16    3.764706         12996.0
1993.0              5         10.90    3.642857          2023.0
1994.0              9         44.56    3.861905         16847.0
1995.0              8         26.70    3.854545          6936.0
1996.0     

Q25 — Do highly wishlisted games lead to more sales?

In [55]:
wishlist_sales = (
    merged[["Title", "Wishlist", "Global_Sales"]]
    .dropna()
    .sort_values("Wishlist", ascending=False)
)

print("Wishlist vs Global Sales:")
print(wishlist_sales.head(20).to_string(index=False))

Wishlist vs Global Sales:
         Title  Wishlist  Global_Sales
    Bloodborne    3300.0          2.38
    Bloodborne    3300.0          2.38
    Bloodborne    3300.0          2.38
    God of War    2600.0          4.45
    God of War    2600.0          4.45
    God of War    2600.0          4.45
     Bayonetta    2300.0          1.23
     Bayonetta    2300.0          0.94
     Bayonetta    2300.0          0.94
     Bayonetta    2300.0          1.23
     Bayonetta    2300.0          1.23
     Bayonetta    2300.0          0.94
Dark Souls III    2200.0          0.05
Chrono Trigger    2200.0          1.47
Dark Souls III    2200.0          1.56
Dark Souls III    2200.0          0.35
Chrono Trigger    2200.0          2.31
Chrono Trigger    2200.0          1.47
Chrono Trigger    2200.0          2.31
Dark Souls III    2200.0          0.05


In [56]:
# Correlation between wishlist and global sales
correlation = merged["Wishlist"].corr(merged["Global_Sales"])

print("Wishlist vs Global Sales Correlation:", correlation)

# Create wishlist groups
merged["Wishlist_Level"] = pd.qcut(
    merged["Wishlist"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

wishlist_analysis = (
    merged.groupby("Wishlist_Level", observed=True)
    .agg(
        Average_Wishlist=("Wishlist", "mean"),
        Average_Global_Sales=("Global_Sales", "mean"),
        Total_Global_Sales=("Global_Sales", "sum"),
        Game_Count=("Title", "nunique")
    )
    .reset_index()
)

print("\nWishlist Level vs Sales:")
print(wishlist_analysis.to_string(index=False))

Wishlist vs Global Sales Correlation: -0.05955234704945692

Wishlist Level vs Sales:
Wishlist_Level  Average_Wishlist  Average_Global_Sales  Total_Global_Sales  Game_Count
           Low        111.053254              2.940000              993.72         130
        Medium        317.817910              3.007104             1007.38         141
          High        602.220896              2.260776              757.36         107
     Very High       1354.026786              2.147738              721.64          92


Q26 — Which genres have the highest engagement but lowest sales?

In [57]:
genre_engagement_sales = (
    merged.groupby("Genre")
    .agg(
        Average_Plays=("Plays", "mean"),
        Average_Global_Sales=("Global_Sales", "mean"),
        Total_Global_Sales=("Global_Sales", "sum"),
        Game_Count=("Title", "nunique")
    )
    .reset_index()
)

genre_engagement_sales = genre_engagement_sales.sort_values(
    "Average_Plays",
    ascending=False
)

print("Genre Engagement vs Sales:")
print(genre_engagement_sales.to_string(index=False))

Genre Engagement vs Sales:
       Genre  Average_Plays  Average_Global_Sales  Total_Global_Sales  Game_Count
        Misc   15818.367347              2.732041              133.87          10
     Shooter    9694.642857              2.377143              532.48          60
      Action    8801.050661              2.115661              960.51         129
    Platform    8129.496933              3.945337              643.09          75
Role-Playing    8049.808411              1.854346              396.83          82
      Sports    7644.656250              8.737500              279.60           7
   Adventure    7276.701149              0.747586               65.04          30
      Racing    5984.615385              7.615769              198.01          13
    Strategy    5372.941176              0.777647               13.22          11
    Fighting    5171.428571              2.072857               87.06          19
      Puzzle    4371.428571              6.444286              135.33  

Q27. Do highly listed games (Wishlist/Backlogs) correlate with better ratings?

In [58]:
# Create total listing/interest
merged["Total_Interest"] = merged["Wishlist"] + merged["Backlogs"]

# Correlation with rating
correlation = merged["Total_Interest"].corr(merged["Rating"])

print("Total Interest vs Rating Correlation:", correlation)

# Divide games into 4 interest levels
merged["Interest_Level"] = pd.qcut(
    merged["Total_Interest"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

interest_rating = (
    merged.groupby("Interest_Level", observed=True)
    .agg(
        Average_Interest=("Total_Interest", "mean"),
        Average_Rating=("Rating", "mean"),
        Game_Count=("Title", "nunique")
    )
    .reset_index()
)

print("\nInterest Level vs Rating:")
print(interest_rating.to_string(index=False))

Total Interest vs Rating Correlation: 0.5164494843261784

Interest Level vs Rating:
Interest_Level  Average_Interest  Average_Rating  Game_Count
           Low        426.387097        3.352493         140
        Medium       1179.586826        3.568563         142
          High       2308.005988        3.716467         112
     Very High       4588.898507        4.037910          78


Q28. How does user engagement differ across genres?

In [59]:
genre_engagement = (
    merged.groupby("Genre")
    .agg(
        Average_Plays=("Plays", "mean"),
        Average_Playing=("Playing", "mean"),
        Average_Backlogs=("Backlogs", "mean"),
        Average_Wishlist=("Wishlist", "mean"),
        Game_Count=("Title", "nunique")
    )
    .reset_index()
    .sort_values("Average_Plays", ascending=False)
)

print("Engagement by Genre:")
print(genre_engagement.to_string(index=False))

Engagement by Genre:
       Genre  Average_Plays  Average_Playing  Average_Backlogs  Average_Wishlist  Game_Count
        Misc   15818.367347       752.408163        663.510204        176.285714          10
     Shooter    9694.642857       139.575893       1502.446429        492.656250          60
      Action    8801.050661       196.878855       1772.486784        734.572687         129
    Platform    8129.496933       116.619632       1286.981595        469.883436          75
Role-Playing    8049.808411       310.214953       2279.738318        920.266355          82
      Sports    7644.656250       167.593750        283.218750         66.312500           7
   Adventure    7276.701149       140.666667       1402.275862        608.505747          30
      Racing    5984.615385        26.807692        296.346154        121.961538          13
    Strategy    5372.941176        65.176471        606.647059        230.058824          11
    Fighting    5171.428571        24.333333     

Q29. Which are the top-performing combinations of Genre + Platform?

In [60]:
genre_platform = (
    merged.groupby(["Genre", "Platform"])
    .agg(
        Global_Sales=("Global_Sales", "sum"),
        Game_Count=("Title", "nunique"),
        Average_Sales=("Global_Sales", "mean")
    )
    .reset_index()
    .sort_values("Global_Sales", ascending=False)
)

print("Top Genre + Platform Combinations:")
print(genre_platform.head(20).to_string(index=False))

Top Genre + Platform Combinations:
       Genre Platform  Global_Sales  Game_Count  Average_Sales
      Sports      Wii        251.01           4      41.835000
      Action      PS3        240.01          52       3.077051
     Shooter     X360        220.97          34       4.803696
      Action     X360        173.85          49       2.519565
     Shooter      PS3        147.47          27       3.985676
      Action      PS2        143.35          32       2.986458
    Platform      Wii        137.02           9       8.563750
      Racing      Wii        107.94           2      26.985000
    Platform      NES        107.10           8       9.736364
    Platform       DS        105.32           9       7.522857
      Puzzle       GB         90.78           1      30.260000
      Action      PS4         82.36          17       2.941429
Role-Playing      PS3         81.32          28       1.807111
Role-Playing     X360         79.20          18       2.828571
      Action       P

Q30. What does a regional sales heatmap by genre reveal?

In [61]:
regional_genre = (
    merged.groupby("Genre")
    .agg(
        NA_Sales=("NA_Sales", "sum"),
        EU_Sales=("EU_Sales", "sum"),
        JP_Sales=("JP_Sales", "sum"),
        Other_Sales=("Other_Sales", "sum")
    )
    .reset_index()
)

print("Regional Sales by Genre:")
print(regional_genre.to_string(index=False))

Regional Sales by Genre:
       Genre  NA_Sales  EU_Sales  JP_Sales  Other_Sales
      Action    458.03    309.29     70.80       121.94
   Adventure     28.90     20.97      8.31         6.93
    Fighting     45.59     18.69     15.71         7.12
        Misc     67.19     41.27      9.33        16.20
    Platform    335.96    158.29    109.34        39.46
      Puzzle     93.93     15.44     22.31         3.68
      Racing     90.87     66.88     24.49        15.84
Role-Playing    167.64    113.19     81.21        34.79
     Shooter    285.37    176.70     12.32        58.01
  Simulation     12.05      9.77     11.20         2.08
      Sports    135.94    101.85     12.26        29.59
    Strategy      5.75      4.59      2.13         0.78
